# 09 — Bygg din egen analyzer (community tutorial)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_100-Custom-Analyzer.ipynb)

**Syfte:** Tutorial för dig som vill bidra med en ny analysmodul till ImintEngine och därmed till SDL 3.0-ekosystemet. Följer ansökans åtagande om **"inclusive collaboration with a wide spectrum of industry and academia stakeholders"**.

**Målgrupp:**
- Masterstudenter (SU, LTU, Stockholms universitet)
- Doktorander inom remote sensing / geospatial AI
- Kommersiella partners (Maxar, PandionAI) som vill bidra med algoritmer
- Myndighetsanalytiker (Skogsstyrelsen, SMHI, Naturvårdsverket)

**Vad du lär dig:**
1. ImintEngines `BaseAnalyzer`-gränssnitt
2. Hur `AnalysisResult` struktureras
3. Registrering via `ANALYZER_REGISTRY`
4. Konfiguration via `analyzers.yaml`
5. Best practices för tester och dokumentation

**Licens:** CC0 1.0 (notebook)

## 1. Arkitekturöversikt

```
Executor (local / colonyos / airflow)
    ↓ bygger IMINTJob med coords + date
    ↓
run_job(job) → körkörs för varje enabled analyzer i registry
    ↓
    AnalyzerA.analyze(rgb, bands, meta) → AnalysisResult
    AnalyzerB.analyze(...)              → AnalysisResult
    ↓
IMINTResult (dict av alla analyses)
    ↓
Exporters (PNG, GeoTIFF, GeoJSON, HTML)
```

**Nyckelkontrakt:** En analyzer tar emot redan hämtad data (RGB + spektralband + meta) och returnerar ett `AnalysisResult`. Den bryr sig inte om **var** data kommer från eller **vart** resultatet går.

## 2. Minsta möjliga analyzer — NDRE (vegetation stress)

Vi bygger en analyzer som beräknar NDRE (Normalized Difference Red Edge):

$$\text{NDRE} = \frac{B_{8A} - B_5}{B_{8A} + B_5}$$

NDRE är känsligare än NDVI för tidig vegetationsstress — används i precisionsjordbruk.

In [ ]:
# Så här ser en analyzer ut:

analyzer_code = '''
# imint/analyzers/ndre.py
from __future__ import annotations
import numpy as np
from .base import BaseAnalyzer, AnalysisResult


class NDREAnalyzer(BaseAnalyzer):
    """
    NDRE (Normalized Difference Red Edge) — vegetation stress index.

    NDRE = (B8A - B5) / (B8A + B5)

    Sensitive to early vegetation stress — useful in precision
    agriculture for detecting water/nutrient deficiencies before
    they are visible in NDVI.
    """

    name = "ndre"

    def __init__(self, stress_threshold: float = 0.2):
        self.stress_threshold = stress_threshold

    def analyze(self, rgb, bands, meta) -> AnalysisResult:
        b5 = bands["B05"].astype(float)
        b8a = bands["B8A"].astype(float)

        denom = b8a + b5
        denom[denom == 0] = 1e-10  # avoid div by zero
        ndre = (b8a - b5) / denom

        stress_mask = ndre < self.stress_threshold
        stress_fraction = float(stress_mask.mean())

        return AnalysisResult(
            name=self.name,
            data={
                "ndre": ndre,
                "stress_mask": stress_mask,
                "stress_fraction": stress_fraction,
            },
            metadata={
                "threshold": self.stress_threshold,
                "interpretation": "stress_fraction > 0.3 = investigate",
            },
        )
'''

print(analyzer_code)

## 3. Registrera analyzern

Lägg till i `imint/engine.py`:

```python
from .analyzers.ndre import NDREAnalyzer

ANALYZER_REGISTRY = {
    "spectral": SpectralAnalyzer,
    "change_detection": ChangeDetectionAnalyzer,
    ...
    "ndre": NDREAnalyzer,   # <-- lägg till här
}
```

Lägg till konfiguration i `config/analyzers.yaml`:

```yaml
ndre:
  enabled: true
  stress_threshold: 0.2    # NDRE < detta = vegetation stress
```

Det är hela integrationen — tre filer, ~40 rader kod.

## 4. Skriv ett test

`tests/test_ndre.py`:

In [ ]:
test_code = '''
# tests/test_ndre.py
import numpy as np
from imint.analyzers.ndre import NDREAnalyzer


def test_ndre_healthy_vegetation():
    """Healthy vegetation has high NDRE (B8A >> B5)."""
    rgb = np.zeros((100, 100, 3), dtype=np.uint8)
    bands = {
        "B05": np.full((100, 100), 0.1),
        "B8A": np.full((100, 100), 0.5),
    }
    meta = {}

    analyzer = NDREAnalyzer(stress_threshold=0.2)
    result = analyzer.analyze(rgb, bands, meta)

    # Expect mostly healthy → low stress fraction
    assert result.data["stress_fraction"] < 0.01
    assert result.data["ndre"].mean() > 0.6


def test_ndre_stressed_vegetation():
    """Stressed vegetation has low NDRE."""
    bands = {
        "B05": np.full((100, 100), 0.3),
        "B8A": np.full((100, 100), 0.35),
    }
    analyzer = NDREAnalyzer(stress_threshold=0.2)
    result = analyzer.analyze(np.zeros((100, 100, 3), dtype=np.uint8), bands, {})

    # Expect mostly stressed → high stress fraction
    assert result.data["stress_fraction"] > 0.99
'''

print(test_code)

## 5. Kör den nya analyzern

In [ ]:
# När analyzern är implementerad i ImintEngine kan du köra den som alla andra:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt

AOI = {"west": 13.2, "south": 55.8, "east": 13.4, "north": 55.9}
DATE = "2024-07-15"

executor = LocalExecutor(output_dir="outputs/ndre_demo")
result = executor.execute(date=DATE, coords=AOI)

if "ndre" in result.analyses:
    ndre_data = result.analyses["ndre"].data
    print(f"Stress fraction: {ndre_data['stress_fraction']:.1%}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(ndre_data["ndre"], cmap="RdYlGn", vmin=-1, vmax=1)
    axes[0].set_title("NDRE")
    axes[0].axis("off")
    axes[1].imshow(ndre_data["stress_mask"], cmap="Reds")
    axes[1].set_title(f"Stress-mask (threshold={ndre_data.get('threshold', 0.2)})")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

## 6. Checklist för PR

Innan du skickar en pull request till [ImintEngine](https://github.com/TobiasEdman/imintengine):

- [ ] Ny fil: `imint/analyzers/<namn>.py` — subklassar `BaseAnalyzer`
- [ ] Registrerad i `imint/engine.py` i `ANALYZER_REGISTRY`
- [ ] Konfigurationsblock i `config/analyzers.yaml`
- [ ] Test: `tests/test_<namn>.py` — minst 2 testfall (healthy + edge case)
- [ ] Docstring — förklara algoritm, input-krav, output-format
- [ ] Referenser till vetenskaplig litteratur om tillämpligt
- [ ] Uppdatera `README.md` — lägg till i analyzer-tabellen
- [ ] Licenskontroll — om du använder modellvikter, dokumentera licens i `THIRD_PARTY_LICENSES.md`

## 7. Exempel på analyzers att bidra med

Som inspiration — domäner där community-bidrag är välkomna:

| Område | Exempel på analyzer |
|--------|---------------------|
| **Klimatanpassning** | Översvämningsdetektion via Sentinel-1 SAR |
| **Skogsbruk** | Barkborreskadeindex (BSI) för granskog |
| **Jordbruk** | LAI (Leaf Area Index) från röd-kant |
| **Stad/infrastruktur** | Byggnadsändringar via change detection |
| **Miljö** | Alg-blomningsdetektion (OC3/OC4) i Östersjön |
| **Försvar** | InSAR-baserad markrörelse (gruvor, broar) |

## 8. Nästa steg

- Diskutera din idé i [space-data-lab GitHub Discussions](https://github.com/TobiasEdman/space-data-lab/discussions)
- Forka [ImintEngine](https://github.com/TobiasEdman/imintengine) och prototypa
- Skicka PR — review görs av RISE/AI Sweden/LTU-teamet
- Godkända analyzers deployas automatiskt till Digital Earth Sweden

### Länkar
- [ImintEngine source](https://github.com/TobiasEdman/imintengine)
- [ImintEngine/imint/analyzers/base.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/base.py) — BaseAnalyzer-kontrakt
- [DES community repo](https://github.com/DigitalEarthSweden/digital-earth-sweden-community)
- [SDL 3.0 ansökan](../../docs/sdl3-source/1.%20Ans%C3%B6kan%20och%20offert/sdl3.0-ans%C3%B6kan.pdf)